In [16]:
import os
import torch
import sys
sys.path.insert(0, "/home/kmercad/mamba_har_2/JEPA_adaptation")
from MambaSSL_JEPA_Model import MambaJEPA, HARMambaConfig, MambaDownstreamClassifier

In [11]:
cd JEPA_adaptation

/home/kmercad/mamba_har_2/JEPA_adaptation


In [2]:
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True NVIDIA A100-SXM4-80GB


In [3]:
# OPPORTUNITY input:     [B, 90, 45]
# Conv1d + BN + GELU:    [B, 90, 384] -> obtaining 90 latent tokens (tokenizer)
# Mamba backbone:        [B, 90, 384]


1. Mask the context input
2. Context encoder forward
3. Target encoder forward on clean input
4. Predictor forward
5. Calculate latent loss
6. loss.backward()
7. optimizer.step()
8. EMA update of target encoder

SyntaxError: invalid syntax (586768327.py, line 6)

In [49]:
def masking_algorithm_targets(tokens, mask_ratio: float = 0.25, t_l: int = 3, generator = None)-> tuple[torch.Tensor, torch.Tensor, list[list[list[int]]]] :
    B, Lw, d_m =  tokens.shape # [32, 18, 385]

    # ---- SPAN: contiguous timestep t_l = 3, mask shape [B, Lw] ----
    n_blocks = max(1, round(Lw * mask_ratio / t_l))   # how many whole t_l-blocks fit the request
    target_current = n_blocks * t_l

    #create empty mask
    mask_targets = torch.zeros((B, Lw), dtype=torch.bool, device=tokens.device)
    target_blocks = []

    for b in range(B):
        I = set()
        counter = 0
        blocks_b = []

        while counter < target_current:
            valid_start = [] # find valid starting indices

            for index in range(Lw - t_l + 1):
                candidates = set(range(index, index + t_l)) # create candidates
                if not(candidates & I):
                    valid_start.append(index)
            if len(valid_start) == 0:
                raise RuntimeError(f"Cannot place span of {t_l} tokens")
                
            pick_indices = torch.randint(0, len(valid_start), (1,), generator = generator).item() #sample rnd value
            candidates = list(range(valid_start[pick_indices], valid_start[pick_indices] + t_l)) # create candidates

            blocks_b.append(candidates) # store the indices of the masked blocks
            I.update(candidates) # create idx for that window
            counter = counter + t_l
        target_blocks.append(blocks_b)
        mask_targets[b, list(I)] = True # need ALL the masked values for the loss.

    context_input = tokens.clone()
    context_input[mask_targets] = 0
    return context_input, mask_targets, target_blocks


In [50]:
device = "cuda" if torch.cuda.is_available() else "cpu"
test_input = torch.rand((2, 18, 9)).to(device)
print(test_input.shape)

torch.Size([2, 18, 9])


In [52]:
context, mask, target = masking_algorithm_targets(test_input)
print(context[1])

tensor([[0.6749, 0.7849, 0.9062, 0.1192, 0.6345, 0.5542, 0.1564, 0.2680, 0.2273],
        [0.4450, 0.5845, 0.7194, 0.8103, 0.3501, 0.6099, 0.1655, 0.3435, 0.1086],
        [0.1023, 0.4389, 0.9519, 0.9449, 0.0333, 0.4668, 0.5654, 0.2848, 0.4053],
        [0.4418, 0.7726, 0.3840, 0.3098, 0.5007, 0.8256, 0.5094, 0.6449, 0.3097],
        [0.2824, 0.1175, 0.6894, 0.3821, 0.3078, 0.6934, 0.2937, 0.9291, 0.2392],
        [0.9045, 0.0597, 0.5348, 0.6306, 0.2052, 0.6794, 0.7803, 0.3327, 0.0527],
        [0.7560, 0.5451, 0.4929, 0.6787, 0.0692, 0.3444, 0.2017, 0.9712, 0.3429],
        [0.3860, 0.6080, 0.8648, 0.9199, 0.8365, 0.7783, 0.9662, 0.8251, 0.6888],
        [0.7018, 0.4194, 0.4595, 0.1599, 0.5790, 0.7777, 0.8429, 0.5678, 0.6268],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3148,

In [ ]:

config = HARMambaConfig()
model = MambaJEPA(config, mask_ratio = 0.25).to(device, non_blocking = True)

B, L, Features = 2, 90, config.num_sensor_features
tensor_ = torch.rand(B, L, Features).to(device)
embeddings_out = model(tensor_)
print("output jepa:", embeddings_out.shape)
model.update_target_encoder(momentum = 0.996)
print("EMA OK") 

target_embeddings, context_embeddings = masking_algorithm_embeddings(embeddings_out, "S", mask_ratio=0.15, Lspan=5)


